# Conway Game of Life

## Cod sursă

```py
class LifeStrategy:

    def next_state(self, cell, env):
        raise NotImplementedError


"""
1. Any live cell with two or three live neighbours lives on to the next generation.
2. Any live cell with fewer than two live neighbours dies, as if by underpopulation.
3. Any live cell with more than three live neighbours dies, as if by overpopulation.
4. Any dead cell with exactly three live neighbours becomes a live cell, as if by reproduction.
"""
class ConwayStrategy(LifeStrategy):
    def next_state(self, cell, env):
        neighbors = cell.perceive(env)
        alive_neighbors = sum(n.state for n in neighbors)

        if cell.state == 1 and alive_neighbors in (2, 3):
            return 1
        elif cell.state == 1 and (alive_neighbors > 3 or alive_neighbors < 2):
            return 0
        elif cell.state == 0 and alive_neighbors == 3:
            return 1
        else:
            return 0


class HighLifeStrategy(LifeStrategy):
    def next_state(self, cell, env):
        neighbors = cell.perceive(env)
        alive_neighbors = sum(n.state for n in neighbors)

        if cell.state == 1 and alive_neighbors in (2, 3):
            return 1
        elif cell.state == 1 and (alive_neighbors > 3 or alive_neighbors < 2):
            return 0
        elif cell.state == 0 and alive_neighbors in (3, 6):
            return 1
        else:
            return 0
 ```

- Folosit pentru a testa diverse strategi în cadrul ciclului de viață al unei celule (bazat pe starea celulelor vecine)

### neighbour.py



```py
class NeighbourFinder:
    def find_neighbours(self, x, y, env):
        """
        Returns the neighbour for a given position in an environment
        :param x: The horizontal position
        :param y: The vertical position
        :param env: The environment variable (of type Environment)
        :return: A list of neighbours
        """
        raise NotImplementedError


class StandardNeighbourFinder(NeighbourFinder):
    def find_neighbours(self, x, y, env):
        directions = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
        neighbors = []

        for dx, dy in directions:
            nx, ny = x + dx, y + dy

            if 0 <= nx < env.width and 0 <= ny < env.height:
                neighbors.append(env.grid[nx][ny])

        return neighbors


class ToroidNeighbourFinder(NeighbourFinder):
    """
    Provides the ability to find neighbours while in edges (by extending the existing map around the other side)
    """
    def find_neighbours(self, x, y, env):
        directions = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
        neighbors = []

        for dx, dy in directions:
            nx, ny = (x + dx) % env.width, (y + dy) % env.height

            neighbors.append(env.grid[nx][ny])

        return neighbors
```




- Folosit pentru metoda de extragere a vecinilor unei celule


### cell.py

```py
import conway.conway.strategy as s


class Cell:
    def __init__(self, x, y, state = 0, strategy = None):
        self.x = x
        self.y = y
        self.state = state
        self.strategy = strategy or s.ConwayStrategy()

    def perceive(self, env):
        """Gathers information from the environment for the cell (the cell neighbours)
        :param env: Environment object (The grid)
        """
        return env.get_neighbors(self.x, self.y)


    def decide_next_state(self, env):
        """Computes the next state of the cell based on the cell life strategy
        :param env: Environment object (The grid)
        """
        return self.strategy.next_state(self, env)

    def __str__(self):
        return str(self.state)
```




- Folosit pentru a reprezenta obiectual o celulă (se leagă de mediu prin metoda `perceive`, cerând mediului informații despre vecinii săi

### environment.py


```py
import random
from conway.conway.cell import Cell
from conway.conway.neighbour import StandardNeighbourFinder


class Environment:
    def __init__(self, width, height, neighbour_finder=None):
        self.width = width
        self.height = height
        self.grid = [[Cell(x, y, random.randint(0, 1)) for x in range(height)] for y in range(width)]
        self.finder = neighbour_finder or StandardNeighbourFinder()


    def get_neighbors(self, x, y):
        """Returns the neighbors of the cell at (x, y).
        :param x: x coordinate
        :param y: y coordinate"""
        return self.finder.find_neighbours(x, y, self)

    def step(self):
        """Updates the environment based on the current cell states."""
        next_states = [
            [cell.decide_next_state(self) for cell in row]
            for row in self.grid
        ]
        for x, row in enumerate(self.grid):
            for y, cell in enumerate(row):
                cell.state = next_states[x][y]
```

- Reprezintă punctul central în care se află celule, oferind acestora abilitatea de a afla informații despre vecinii lor

### simulator.py

```py
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation

from conway.conway.cell import Cell
from conway.conway.dir_util import make_file_dir_if_not_exist
from conway.conway.environment import Environment
from conway.conway.strategy import ConwayStrategy
from conway.conway.neighbour import StandardNeighbourFinder


class ConwaySimulator:
    def __init__(self, width, height, life_strategy=ConwayStrategy(), initial_state=None,
                 neighbour_finder=StandardNeighbourFinder()):
        self.width = width
        self.height = height
        self.strategy = life_strategy
        self.environment = Environment(width, height, neighbour_finder)
        self.initialize_cells(initial_state)

        # Animation

        self.fig, self.ax = plt.subplots()

        states = self.__get_cells_states()

        self.mat = self.ax.matshow(states, vmin=0, vmax=1, origin='upper')
        self.text = self.ax.text(0.5, 0.5, "", bbox={'facecolor': 'white', 'alpha': 0.5, 'pad': 2})

    def initialize_cells(self, initial_state):

        if initial_state is not None:

            for x in range(self.height):
                for y in range(self.width):
                    state = int(initial_state[x][y])
                    self.environment.grid[x][y] = Cell(x, y, state)
                    self.environment.grid[x][y].strategy = self.strategy

    def __update(self, data):

        if data != 0:
            self.environment.step()
        states = self.__get_cells_states()

        self.mat.set_data(states)
        self.text.set_text(f'Iteration {data}')
        return [self.mat]

    def __get_cells_states(self):
        states = np.array([[cell.state for cell in row] for row in self.environment.grid])
        return states

    def run(self, frame_interval:int=100, iterations:int=40, filename:str=None) -> FuncAnimation:
        """Runs and returns the simulation for the given number of iterations, spacing the frames at the given time interval. If the filename parameter is not null, it will be saved to that file.
        :param frame_interval The interval between frames
        :param iterations The number of iterations to run the simulation for
        :param filename The name of the file where to save the simulation video (can include directories)"""
        animation = FuncAnimation(self.fig, self.__update, interval=frame_interval, save_count=iterations)

        if filename is not None:
            make_file_dir_if_not_exist(filename)
            animation.save(filename)

        return animation
```


- Reprezintă o abstractizare a jocului menită să fie utilizată de utilizator pentru a putea rula simularea (print metoda `run`)